In [ ]:
from datasets import load_dataset, Audio

TARGET_HOURS = 1.5
TARGET_SECONDS = TARGET_HOURS * 3600


def take_hours(dataset, target_seconds):
    selected = []
    total = 0

    for example in dataset:
        duration = len(example["audio"]["array"]) / example["audio"]["sampling_rate"]

        selected.append(example)
        total += duration

        if total >= target_seconds:
            break

    from datasets import Dataset
    return Dataset.from_list(selected)


# Русский
ru = load_dataset(
    "mort666/cv_corpus_v22",
    "ru",
    split="train",
    trust_remote_code=True
)

# Английский
en = load_dataset(
    "mort666/cv_corpus_v22",
    "en",
    split="train",
    trust_remote_code=True
)

/home/fedosdan2/miniconda3/envs/jarvis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating invalidated split: 100%|██████████| 306854/306854 [00:57<00:00, 5291.16 examples/s]


ImportError: To support decoding audio files, please install 'librosa' and 'soundfile'.

In [1]:
from datasets import load_dataset

ru = load_dataset(
    "mort666/cv_corpus_v22",
    "ru",
    split="train",
    trust_remote_code=True
)

en = load_dataset(
    "mort666/cv_corpus_v22",
    "en",
    split="train",
    trust_remote_code=True
)

print(len(ru))
print(len(en))

/home/fedosdan2/miniconda3/envs/jarvis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


26654
1138760


In [2]:
print(ru)
print(ru.column_names)
print(ru[0])

print(en)
print(en.column_names)
print(en[0])

Dataset({
    features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment', 'variant'],
    num_rows: 26654
})
['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment', 'variant']
{'client_id': '299f0dd85f2861f679e92991dc15e9adbdcc7cb8ce7e6df075f462a56ec7664f2f5e37671d88006d525e3c5454dd5e1258bf38e85df96b1356861fadbf259999', 'path': '/home/skapp/.cache/huggingface/datasets/downloads/extracted/555859a6fe4019ba20008da757456976387423e516b4fdba9a773db6e49809fe/ru_train_0/common_voice_ru_18849051.mp3', 'audio': {'path': 'common_voice_ru_18849051.mp3', 'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(312192,)), 'sampling_rate': 48000}, 'sentence': 'В особенности, я повторяю, мы отводим высочайший приоритет ядерному разоружению.', 'up_votes': 2, 'down_votes': 0, 'age': 'twenties', 'gender': 'male_masculine', 'accent': '', 'locale': 'ru', 'segment': '', 'variant': ''}


In [3]:
from collections import Counter

print("RU speakers:", len(set(ru["client_id"])))
print("EN speakers:", len(set(en["client_id"])))

print("RU top speakers:")
print(Counter(ru["client_id"]).most_common(10))

print("EN top speakers:")
print(Counter(en["client_id"]).most_common(10))

RU speakers: 326
EN speakers: 39267
RU top speakers:
[('8a127fc948673d60d84fe655fe2065747e0f45e96a5150470517a71b4e44b29ca53460963a85dd976c691fd1a25a17ad70e601d68c4542bfc7403ced29f7574f', 1037), ('299f0dd85f2861f679e92991dc15e9adbdcc7cb8ce7e6df075f462a56ec7664f2f5e37671d88006d525e3c5454dd5e1258bf38e85df96b1356861fadbf259999', 956), ('58eda5ae08bee1863e072e53a76671e79afc578897789d43db913670084f0b6dbee4f6c2577928b0de8baa7257611b9cea9be6f58a57925a6edd6ed566481001', 491), ('1a0eea2cb817ad4f14b1465947dd92a6fabdd3a50bb9a62f6675cce6900845b7b709c9be4d77ac2b341c02efa5255cddcb315f13fad7e5feb00ab6a112c4aadd', 348), ('932bcfa705ccd94761da744394963590a67ca3e7bc4a3495747a84dc70ab567c859c94721b8c76123b503c94da005bd9652769ce1926f8c7e69cfaaef7ac1abb', 336), ('ce4367b66040fc7d783c63c5c2d73fe58cb2038a590c4aeb96d9295f4e48143f0cd30db40b74202712393f3e8fd67e76f69717a632e8062dae40a39e758b213c', 323), ('8213a0f79acb074bf5427027efa90e01391d8709700f103d434a0831e9232e7c83551fff5f5e2bf4e65128b0432eeec7292ecbfaa0c62

In [10]:
from collections import defaultdict
import random

random.seed(42)


def select_diverse_indices_with_minimum(
    dataset,
    max_speakers=200,
    max_per_speaker=10,
    min_per_speaker=10,
):
    speaker_indices = defaultdict(list)

    for idx, client_id in enumerate(dataset["client_id"]):
        speaker_indices[client_id].append(idx)

    # Берём только спикеров, у которых достаточно записей
    eligible = [
        (client_id, indices)
        for client_id, indices in speaker_indices.items()
        if len(indices) >= min_per_speaker
    ]

    # Случайно выбираем нужное количество спикеров
    random.shuffle(eligible)
    selected = eligible[:max_speakers]

    indices = []
    speakers = {}

    for client_id, speaker_indices_list in selected:

        chosen = speaker_indices_list[:max_per_speaker]

        speakers[client_id] = chosen
        indices.extend(chosen)

    random.shuffle(indices)

    return indices, speakers

In [ ]:
RU_SPEAKERS = 300
EN_SPEAKERS = 200

MAX_UTTERANCES_PER_SPEAKER = 10

en_indices, en_speakers = select_diverse_indices_with_minimum(
    en,
    max_speakers=200,
    max_per_speaker=10,
    min_per_speaker=10,
)

print("EN speakers:", len(en_speakers))
print("EN utterances:", len(en_indices))

ru_indices, ru_speakers = select_diverse_indices_with_minimum(
    ru,
    max_speakers=200,
    max_per_speaker=10,
    min_per_speaker=10,
)

print("RU speakers:", len(ru_speakers))
print("RU utterances:", len(ru_indices))


EN speakers: 200
EN utterances: 2000
RU speakers: 200
RU utterances: 2000


In [18]:
from pathlib import Path

import soundfile as sf
import librosa
import numpy as np
import csv


TARGET_SR = 16000
TARGET_HOURS = 1.0
TARGET_SECONDS = TARGET_HOURS * 3600


def export_dataset(
    dataset,
    indices,
    speakers,
    output_dir,
    language,
    target_seconds,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    metadata_path = output_dir / "metadata.csv"

    total_seconds = 0.0
    saved = 0

    indices = list(indices)

    with open(metadata_path, "w", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)

        writer.writerow([
            "file",
            "client_id",
            "language",
            "duration",
            "sentence",
        ])

        for idx in indices:

            # ==================================================
            # ВАЖНО:
            # Проверяем ДО загрузки следующего аудио
            # ==================================================
            if total_seconds >= target_seconds:
                break

            example = dataset[idx]

            audio = example["audio"]

            waveform = audio["array"]
            sr = audio["sampling_rate"]

            # mono
            if waveform.ndim > 1:
                waveform = np.mean(waveform, axis=0)

            # float32
            waveform = waveform.astype(np.float32)

            # 16 kHz
            if sr != TARGET_SR:
                waveform = librosa.resample(
                    waveform,
                    orig_sr=sr,
                    target_sr=TARGET_SR,
                )

            duration = len(waveform) / TARGET_SR

            # ==================================================
            # Если весь файл уже не помещается в оставшееся
            # время — НЕ сохраняем его.
            # ==================================================
            remaining = target_seconds - total_seconds

            if duration > remaining:
                break

            # ==================================================
            # Уникальное имя
            # ==================================================
            filename = f"{language}_{saved:06d}.wav"

            filepath = output_dir / filename

            # ==================================================
            # Сохраняем PCM16 WAV
            # ==================================================
            sf.write(
                filepath,
                waveform,
                TARGET_SR,
                subtype="PCM_16",
            )

            # ==================================================
            # Metadata
            # ==================================================
            writer.writerow([
                filename,
                example["client_id"],
                language,
                duration,
                example["sentence"],
            ])

            total_seconds += duration
            saved += 1

            if saved % 100 == 0:
                print(
                    f"{language}: "
                    f"{saved} files, "
                    f"{total_seconds / 3600:.2f} h"
                )

    print()
    print(f"{language} finished")
    print(f"Files: {saved}")
    print(f"Duration: {total_seconds / 3600:.2f} h")
    print(f"Output: {output_dir}")

In [19]:
export_dataset(
    dataset=ru,
    indices=ru_indices,
    speakers=ru_speakers,
    output_dir="/home/fedosdan2/Study/Jarvis/data/impostor/ru",
    language="ru",
    target_seconds=1.0 * 3600,
)

ru: 100 files, 0.15 h
ru: 200 files, 0.30 h
ru: 300 files, 0.45 h
ru: 400 files, 0.60 h
ru: 500 files, 0.75 h
ru: 600 files, 0.90 h

ru finished
Files: 667
Duration: 1.00 h
Output: /home/fedosdan2/Study/Jarvis/data/impostor/ru


In [20]:
export_dataset(
    dataset=en,
    indices=en_indices,
    speakers=en_speakers,
    output_dir="/home/fedosdan2/Study/Jarvis/data/impostor/en",
    language="en",
    target_seconds=1.0 * 3600,
)

en: 100 files, 0.16 h
en: 200 files, 0.32 h
en: 300 files, 0.49 h
en: 400 files, 0.65 h
en: 500 files, 0.82 h
en: 600 files, 0.98 h

en finished
Files: 609
Duration: 1.00 h
Output: /home/fedosdan2/Study/Jarvis/data/impostor/en


In [ ]:
from pathlib import Path
import csv

import numpy as np
import soundfile as sf


RAW_DIR = Path("/home/fedosdan2/Music")
OUTPUT_DIR = Path("/home/fedosdan2/Study/Jarvis/data/own")

TARGET_SR = 16000

CHUNK_SECONDS = 5
CHUNK_SAMPLES = TARGET_SR * CHUNK_SECONDS


def process_sessions():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    metadata_path = OUTPUT_DIR / "metadata.csv"

    wav_files = sorted(RAW_DIR.glob("session_*.wav"))

    if not wav_files:
        raise RuntimeError(f"Не найдено записей в {RAW_DIR}")

    print(f"Найдено сессий: {len(wav_files)}")

    saved_ru = 0
    saved_en = 0
    
    with open(metadata_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        writer.writerow([
            "file",
            "client_id",
            "language",
            "duration",
            "sentence",
        ])

        for wav_path in wav_files:
            print(f"\nОбрабатываю: {wav_path.name}")
            
            info = sf.info(wav_path)

            # Загружаем запись
            waveform, sr = sf.read(
                wav_path,
                dtype="float32",
            )

            # Режем на куски по 5 секунд
            total_samples = len(waveform)
            num_chunks = total_samples // CHUNK_SAMPLES

            for chunk_idx in range(num_chunks):
                start = chunk_idx * CHUNK_SAMPLES
                end = start + CHUNK_SAMPLES

                chunk = waveform[start:end]
                
                if not 'en' in wav_path.name:
                    filename = (f"ru_{saved_ru:06d}.wav")
                    saved_ru += 1
                    
                    output_path = OUTPUT_DIR / 'ru' / filename
                    
                    sf.write(output_path, chunk, TARGET_SR, subtype="PCM_16")
    
                    writer.writerow([filename, "own", "ru", CHUNK_SECONDS, "",])
                else: 
                    filename = (f"en_{saved_en:06d}.wav")
                    saved_en += 1
                    
                    output_path = OUTPUT_DIR / 'en' / filename
                    
                    sf.write(output_path, chunk, TARGET_SR, subtype="PCM_16")
    
                    writer.writerow([filename, "own", "en", CHUNK_SECONDS, "",])




            # Остаток меньше 5 секунд просто выбрасываем
            remainder = total_samples % CHUNK_SAMPLES

            if remainder > 0:
                print(
                    f"  Остаток {remainder / TARGET_SR:.2f} sec "
                    f"не сохраняем"
                )

            print(f"  Создано кусков: {num_chunks}")

    print("\nГотово")
    print(f"Metadata: {metadata_path}")


process_sessions()

In [25]:
RAW_DIR_EN = Path("/home/fedosdan2/Study/Jarvis/data/own/en")
RAW_DIR_RU = Path("/home/fedosdan2/Study/Jarvis/data/own/ru")

ru_len = len(sorted(RAW_DIR_RU.glob("*.wav")))
en_len = len(sorted(RAW_DIR_EN.glob("*.wav")))

print(ru_len, en_len)

601 212
